 # Phase 2 — Exploring DB Train Delays Dataset
 ## Setup

In [10]:
import sys
print(sys.executable)

c:\Users\student\Documents\Projects\db_delays-analysis\.venv\Scripts\python.exe


In [11]:
import pandas as pd
print(pd.__version__)

3.0.2


In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Libraries loaded.")
print("Files in data/:", os.listdir("../data"))


Libraries loaded.
Files in data/: ['DBtrainrides.csv']


## Loading the data

In [13]:
df = pd.read_csv("../data/DBtrainrides.csv")
print("Shape:", df.shape)
df.head()

Shape: (2061357, 20)


,ID,line,path,eva_nr,category,station,state,city,zip,long,lat,arrival_plan,departure_plan,arrival_change,departure_change,arrival_delay_m,departure_delay_m,info,arrival_delay_check,departure_delay_check
0,1573967790757085557-2407072312-14,20,Stolberg(Rheinl)Hbf Gl.44|Eschweiler-St.Jöris|...,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,2024-07-08 00:00:00,2024-07-08 00:01:00,2024-07-08 00:03:00,2024-07-08 00:04:00,3,3,NaN,on_time,on_time
1,349781417030375472-2407080017-1,18,NaN,8000001,2,Aachen Hbf,Nordrhein-Westfalen,Aachen,52064,6.091499,50.767800,NaN,2024-07-08 00:17:00,NaN,NaN,0,0,NaN,on_time,on_time
2,7157250219775883918-2407072120-25,1,Hamm(Westf)Hbf|Kamen|Kamen-Methler|Dortmund-Ku...,8000406,4,Aachen-Rothe Erde,Nordrhein-Westfalen,Aachen,52066,6.116475,50.770202,2024-07-08 00:03:00,2024-07-08 00:04:00,2024-07-08 00:03:00,2024-07-08 00:04:00,0,0,NaN,on_time,on_time
3,349781417030375472-2407080017-2,18,Aachen Hbf,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,NaN,NaN,0,0,NaN,on_time,on_time
4,1983158592123451570-2407080010-3,33,Herzogenrath|Kohlscheid,8000404,5,Aachen West,Nordrhein-Westfalen,Aachen,52072,6.070715,50.780360,2024-07-08 00:20:00,2024-07-08 00:21:00,2024-07-08 00:20:00,2024-07-08 00:21:00,0,0,NaN,on_time,on_time


 ## Understanding columns and missing values

In [14]:
print("=== Column types and non-null counts ===")
df.info()
print()
print("=== Missing values per column ===")
print(df.isnull().sum())
print()
print("=== Unique categories ===")
print("category values:", df["category"].unique())
print()
print("=== Sample of arrival_delay_check values ===")
print(df["arrival_delay_check"].value_counts())

=== Column types and non-null counts ===
<class 'pandas.DataFrame'>
RangeIndex: 2061357 entries, 0 to 2061356
Data columns (total 20 columns):
 #   Column                 Dtype  
---  ------                 -----  
 0   ID                     str    
 1   line                   str    
 2   path                   str    
 3   eva_nr                 int64  
 4   category               int64  
 5   station                str    
 6   state                  str    
 7   city                   str    
 8   zip                    int64  
 9   long                   float64
 10  lat                    float64
 11  arrival_plan           str    
 12  departure_plan         str    
 13  arrival_change         str    
 14  departure_change       str    
 15  arrival_delay_m        int64  
 16  departure_delay_m      int64  
 17  info                   str    
 18  arrival_delay_check    str    
 19  departure_delay_check  str    
dtypes: float64(2), int64(5), str(13)
memory usage: 314.5 MB

===

## Investigating delay distribution

In [15]:
print("=== Cross-tab: delay status vs actual delay minutes ===")
print(df.groupby("arrival_delay_check")["arrival_delay_m"].describe())
print()
print("=== Distribution of arrival_delay_m ===")
print(df["arrival_delay_m"].describe())
print()
print("=== How many rows by delay threshold? ===")
for threshold in [0, 1, 5, 6, 15, 30, 60]:
 count = (df["arrival_delay_m"] > threshold).sum()
 pct = 100 * count / len(df)
 print(f"Delayed > {threshold:3d} min: {count:>10,} rows ({pct:5.2f}%)")

=== Cross-tab: delay status vs actual delay minutes ===
                         count       mean       std  min  25%  50%   75%    max
arrival_delay_check                                                            
delay                 110953.0  11.954756  8.444300  6.0  7.0  9.0  14.0  159.0
on_time              1950404.0   0.563441  1.110545  0.0  0.0  0.0   1.0    5.0

=== Distribution of arrival_delay_m ===
count    2.061357e+06
mean     1.176581e+00
std      3.407859e+00
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      1.000000e+00
max      1.590000e+02
Name: arrival_delay_m, dtype: float64

=== How many rows by delay threshold? ===
Delayed >   0 min:    654,452 rows (31.75%)
Delayed >   1 min:    399,407 rows (19.38%)
Delayed >   5 min:    110,953 rows ( 5.38%)
Delayed >   6 min:     89,003 rows ( 4.32%)
Delayed >  15 min:     21,708 rows ( 1.05%)
Delayed >  30 min:      4,227 rows ( 0.21%)
Delayed >  60 min:        281 rows ( 0.01%)


# Phase 2 — Exploration Summary

## Dataset overview
- 2,061,357 rows × 20 columns
- One row per train arrival/departure event
- ~10% missing arrival data (likely starting stations of routes)
- Train type column is NOT available — adapted Q2 to use station category

## Key early findings
- 95.4% of train stops are within DB's 6-minute "on-time" threshold
- But 31.75% have *some* delay (>0 minutes)
- Median delay = 0 minutes; mean = 1.18 minutes
- Max delay = 159 minutes (heavy right-skewed distribution)

## What still needs cleaning
- Date columns are stored as strings - must convert to datetime
- Decide how to handle rows with missing arrival_plan
- Decide on a single "delay" definition for the dashboard